# Вторичные структуры ДНК у *Eledone cirrhosa* (пункт 4): G-квадруплексы + Z-ДНК (zhunt)

Адаптация reference-тетрадки вторичных структур под локальную машину и геном *Eledone* (Ensembl, имена хромосом `1..26,Z`).

Делает два BED-файла:
- **`results/bed/g4.bed`** — G-квадруплексы: regex на обоих стрендах по верхнему регистру (`G{3,5}`/`C{3,5}`).
- **`results/bed/zhunt.bed`** — Z-ДНК: программа `zhunt3` (Z-HUNT-II, C), порог `z-score > 400`.

**Охват:** весь геном (`CHROMS=None`) — нужно для групповой части, где гены-ортологи могут быть на любой хромосоме. Для индивидуальной таблицы можно ограничиться подмножеством хромосом.

zhunt однопоточный (~1.8 мин/Мб), поэтому каждая хромосома режется на куски и считается параллельно (`ZHUNT_PROC` процессов); на весь геном (~3 Гб) при 96 процессах ≈ час. G4 — быстрый, идёт по всем хромосомам сразу.

> zhunt считает только `a/t/g/c` и **пропускает `N`** → номер строки в `.Z-SCORE` = порядковый номер ATGC-базы. Строим маппинг строка→геномная координата, чтобы координаты в BED были правильные.

## 0. Базовая директория и импорты

In [3]:
%pip install -q biopython
import os, re, subprocess, tempfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict

base_dir = Path("/home/bulat_kharisov/inf")
(base_dir / "results" / "bed").mkdir(parents=True, exist_ok=True)
(base_dir / "scripts").mkdir(parents=True, exist_ok=True)
os.chdir(base_dir)

from Bio import SeqIO
print("base_dir:", base_dir, "| cwd:", os.getcwd())


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
base_dir: /home/bulat_kharisov/inf | cwd: /home/bulat_kharisov/inf


## 1. Параметры

- `CHROMS` — хромосомы (та же, что ZDNABERT); `None` = все.
- `MAX_BP` — bp на хромосому; `None` = вся.
- `G4_PLUS`/`G4_MINUS` — regex квадруплексов на `+`/`-` цепи.
- `ZHUNT_WIN/MIN/MAX` — аргументы zhunt (`12 8 12`), `Z_THRESHOLD` — порог z-score (> 400).
- `ZHUNT_CHUNK`/`ZHUNT_OVERLAP`/`ZHUNT_PROC` — нарезка и параллель для zhunt.

In [ ]:
FASTA_IN  = base_dir / "data" / "genome.fa"
G4_BED    = base_dir / "results" / "bed" / "g4.bed"
ZHUNT_BED = base_dir / "results" / "bed" / "zhunt.bed"

CHROMS  = None         # None = все хромосомы (весь геном); список имён = подмножество
MAX_BP  = None         # None = вся длина хромосомы

# G-квадруплексы (оба стренда, верхний регистр)
G4_PLUS  = re.compile(r"(?:G{3,5}[ACGT]{1,7}){3,}G{3,5}")
G4_MINUS = re.compile(r"(?:C{3,5}[ACGT]{1,7}){3,}C{3,5}")

# zhunt
ZHUNT_WIN, ZHUNT_MIN, ZHUNT_MAX = 12, 8, 12
Z_THRESHOLD   = 400.0       # порог z-score (в плане > 400)
ZHUNT_CHUNK   = 2_000_000   # bp на кусок (zhunt однопоточный -> параллелим кусками)
ZHUNT_OVERLAP = 1_000       # контекст на стыке (>= 2*ZHUNT_MAX), потом отбрасываем
ZHUNT_PROC    = 96          # сколько zhunt-процессов параллельно (<= ядер)
ZHUNT_SRC_URL = "https://raw.githubusercontent.com/diegopenilla/FlaskHunt/master/zhunt3.c"

assert FASTA_IN.exists(), f"нет генома {FASTA_IN}"

## 2. Загрузка последовательностей хромосом (верхний регистр, срез до MAX_BP)

In [5]:
seqs = {}
for rec in SeqIO.parse(str(FASTA_IN), "fasta"):
    if CHROMS is not None and rec.id not in CHROMS:
        continue
    s = str(rec.seq).upper()
    if MAX_BP is not None:
        s = s[:MAX_BP]
    seqs[rec.id] = s
    if CHROMS is not None and len(seqs) == len(CHROMS):
        break
print("загружено:", {k: len(v) for k, v in seqs.items()})

загружено: {'1': 1000000}


## 3. G-квадруплексы -> `results/bed/g4.bed`

`+` цепь — G-богатые мотивы, `-` цепь — C-богатые (квадруплекс на комплементарной цепи). `finditer` — непересекающиеся совпадения.

In [6]:
def find_g4(seq, chrom):
    rows = []
    for strand, pat in (("+", G4_PLUS), ("-", G4_MINUS)):
        for m in pat.finditer(seq):
            rows.append((chrom, m.start(), m.end(), strand))
    return rows

n = 0
with open(G4_BED, "w") as bed:
    for chrom, s in seqs.items():
        rows = find_g4(s, chrom)
        rows.sort(key=lambda r: (r[1], r[2]))
        for c, st, en, strand in rows:
            bed.write(f"{c}\t{st}\t{en}\tG4\t{en - st}\t{strand}\n")
            n += 1
print("готово ->", G4_BED, "| G4-интервалов:", n)

готово -> /home/bulat_kharisov/inf/results/bed/g4.bed | G4-интервалов: 86


## 4. zhunt: скачать исходник и скомпилировать

`zhunt3.c` — версия Z-HUNT-II с serialized I/O. Сборка без `-DUSE_MMAP` (иначе код использует захардкоженный путь).

In [7]:
zhunt_src = base_dir / "scripts" / "zhunt3.c"
zhunt_bin = base_dir / "scripts" / "zhunt3"
if not zhunt_src.exists():
    subprocess.run(["wget", "-q", "-O", str(zhunt_src), ZHUNT_SRC_URL], check=True)
subprocess.run(["gcc", str(zhunt_src), "-o", str(zhunt_bin), "-lm"], check=True)
print("zhunt собран:", zhunt_bin)

zhunt собран: /home/bulat_kharisov/inf/scripts/zhunt3


/home/bulat_kharisov/inf/scripts/zhunt3.c: In function ‘user_regret’:
/home/bulat_kharisov/inf/scripts/zhunt3.c:290:7: warning: implicit declaration of function ‘gets’; did you mean ‘fgets’? [-Wimplicit-function-declaration]
  290 |       gets( tempstr );
      |       ^~~~
      |       fgets
/usr/bin/ld: /tmp/ccxRy3hm.o: in function `user_regret':
zhunt3.c:(.text+0xb21): warning: the `gets' function is dangerous and should not be used.


## 5. zhunt: прогнать по кускам параллельно -> `results/bed/zhunt.bed`

Каждый кусок (+`OVERLAP` для контекста окна) пишется во временный файл из чистых ATGC, прогоняется через `zhunt3`, фильтруется `awk` по 3-му столбцу (z-score). Номер строки -> геномная координата через маппинг (пропуск `N`). Берём только позиции в «своей» зоне куска (`< owned_end`), затем сливаем последовательные позиции в интервалы.

In [8]:
def zhunt_chunk(task):
    chrom, gstart, owned_end, sub = task
    acgt = sum(sub.count(b) for b in "ACGT")
    if acgt == 0:
        return chrom, []
    if acgt == len(sub):                      # нет N -> прямая координата
        seq_clean = sub
        gmap = lambda i: gstart + i
    else:                                     # есть N -> маппинг через сохранённые индексы
        keep = [j for j, b in enumerate(sub) if b in "ACGT"]
        seq_clean = "".join(sub[j] for j in keep)
        gmap = lambda i: gstart + keep[i]
    with tempfile.NamedTemporaryFile("w", suffix=".seq",
                                     dir=str(base_dir / "scripts"), delete=False) as tf:
        tf.write(seq_clean)
        tmp = tf.name
    try:
        subprocess.run([str(zhunt_bin), str(ZHUNT_WIN), str(ZHUNT_MIN), str(ZHUNT_MAX), tmp],
                       check=True, stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
        out = subprocess.run(["awk", "-v", f"t={Z_THRESHOLD}",
                              "NR>1 && $3>t {print NR-2}", tmp + ".Z-SCORE"],
                             check=True, capture_output=True, text=True).stdout
    finally:
        for f in (tmp, tmp + ".Z-SCORE"):
            if os.path.exists(f):
                os.remove(f)
    flagged = []
    for x in out.split():
        p = gmap(int(x))
        if p < owned_end:                     # выбросить хвост из зоны OVERLAP
            flagged.append(p)
    return chrom, flagged

def merge_positions(positions):
    positions = sorted(positions)
    out = []
    if positions:
        st = pr = positions[0]
        for v in positions[1:]:
            if v <= pr + 1:
                pr = v
            else:
                out.append((st, pr + 1)); st = pr = v
        out.append((st, pr + 1))
    return out

tasks = []
for chrom, s in seqs.items():
    L = len(s)
    for st in range(0, L, ZHUNT_CHUNK):
        owned_end = min(st + ZHUNT_CHUNK, L)
        en = min(st + ZHUNT_CHUNK + ZHUNT_OVERLAP, L)
        tasks.append((chrom, st, owned_end, s[st:en]))
print(f"zhunt: кусков {len(tasks)} x ~{ZHUNT_CHUNK // 10**6} Мб | процессов {ZHUNT_PROC}")

by_chrom = defaultdict(list)
done = 0
with ThreadPoolExecutor(max_workers=ZHUNT_PROC) as ex:
    for chrom, flagged in ex.map(zhunt_chunk, tasks):
        by_chrom[chrom].extend(flagged)
        done += 1
        print(f"  {done}/{len(tasks)} кусков", end="\r")

n = 0
with open(ZHUNT_BED, "w") as bed:
    for chrom in sorted(by_chrom):
        for st, en in merge_positions(by_chrom[chrom]):
            bed.write(f"{chrom}\t{st}\t{en}\tZHUNT\t{en - st}\t.\n")
            n += 1
print("\nготово ->", ZHUNT_BED, "| zhunt-интервалов:", n)

zhunt: кусков 1 x ~2 Мб | процессов 96
  1/1 кусков
готово -> /home/bulat_kharisov/inf/results/bed/zhunt.bed | zhunt-интервалов: 760


## Итог

- `results/bed/g4.bed` — G-квадруплексы (BED6, со стрендом).
- `results/bed/zhunt.bed` — Z-ДНК по zhunt (`z-score > 400`).
- G4 и zhunt — по всему геному (для групповой части); ZDNABERT — пилот на chr1 (для индивидуальной таблицы).

Дальше по плану: участки из GFF (пункт 5: exons/introns/promoters/downstream/intergenic) и bedtools intersect + фон (пункт 6).